# Translation Pipeline using Transformers 

In [ ]:
!pip install evaluate
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00


Loading the dataset **English-to-hindi-podcast-translation**

In [31]:
from datasets import load_dataset
import numpy as np
ds = load_dataset("rajuptvs/English-to-hindi-podcast-translation")

Loading the model **mbart-large** for translation task

In [51]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [94]:
print("-"*100)
print(">> Dataset overview")
print(ds)
print("-"*100)
print(ds['train'][0])
print("-"*100)


----------------------------------------------------------------------------------------------------
>> Dataset overview
DatasetDict({
    train: Dataset({
        features: ['video_id', 'English subtitles', 'Hindi subtitles', '__index_level_0__'],
        num_rows: 11427
    })
})
----------------------------------------------------------------------------------------------------
{'video_id': '44iAPrQoYU8', 'English subtitles': 'imposter syndrome how does one move past', 'Hindi subtitles': 'इम्पोस्टर सिंड्रोम से कोई कैसे उबर सकता है, आराम से', '__index_level_0__': 0}
----------------------------------------------------------------------------------------------------


**Preprocessing the Dataset**

In [4]:
ds = ds.remove_columns(['video_id','__index_level_0__'])
ds

DatasetDict({
    train: Dataset({
        features: ['English subtitles', 'Hindi subtitles'],
        num_rows: 11427
    })
})

In [5]:
split_dataset = ds['train'].train_test_split(train_size=0.9,seed=45)
split_dataset['validation'] = split_dataset.pop('test')

In [52]:
max_length = 128
def tokenize_function(dataset):
    english_sentence = [text for text in dataset['English subtitles']]
    hindi_sentence = [text for text in dataset['Hindi subtitles']]
    model_inputs = tokenizer(
        english_sentence,text_target=hindi_sentence,max_length=max_length,truncation=True
    )
    return model_inputs

In [53]:
tokenized_dataset = split_dataset.map(tokenize_function,batched=True,remove_columns=split_dataset['train'].column_names)

Map:   0%|          | 0/10284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1143 [00:00<?, ? examples/s]

In [54]:
from transformers import DataCollatorForSeq2Seq 
DataCollator = DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model)

**Preparing dataloader**

In [ ]:
from torch.utils.data import DataLoader
tokenized_dataset.set_format('torch')

Train_dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size = 8,
    shuffle=True,
    collate_fn = DataCollator
)

Eval_dataloader = DataLoader(
    tokenized_dataset['validation'],
    batch_size = 8,
    shuffle=False,
    collate_fn = DataCollator
)

In [56]:
import torch
EPOCHS = 5 
num_steps_update_per_epochs = len(Train_dataloader)
num_train_steps = EPOCHS * num_steps_update_per_epochs 

optimizer = torch.optim.Adam(model.parameters(),lr=0.00002)

In [57]:
from transformers import get_scheduler 
lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps = num_train_steps
)

In [58]:
import evaluate
metric = evaluate.load("sacrebleu")

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

**Train Loop**

In [59]:
from tqdm.auto import tqdm 
progress_bar = tqdm(range(num_train_steps))
model = model.to('cuda')

for epoch in range(EPOCHS):
    model.train()
    for batch in Train_dataloader:
        batch = batch.to('cuda')
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    model.eval()
    for batch in Eval_dataloader:
        batch = batch.to('cuda')
        with torch.no_grad():
            generated_tokens = model.generate(
                batch['input_ids'],
                attention_mask = batch['attention_mask'],
                max_length = 128,
            )

        labels = batch['labels']
        decoded_preds , decoded_labels = postprocess(generated_tokens,labels)
        metric.add_batch(predictions=decoded_preds,references=decoded_labels)
    results = metric.compute()
    print(f"[{epoch+1}/{EPOCHS}] | BLEU score : {results['score']:.2f}")


  0%|          | 0/6430 [00:00<?, ?it/s]

[1/5] | BLEU score : 28.00
[2/5] | BLEU score : 28.47
[3/5] | BLEU score : 29.43
[4/5] | BLEU score : 28.51
[5/5] | BLEU score : 28.35


# Inference

In [90]:
device = 'cuda' if torch.cuda.is_available() else "cpu"
text = input()
input_tensor = tokenizer(text,max_length=128,truncation=True,return_tensors='pt').to(device)
with torch.no_grad():
    output = model.generate(
        input_tensor['input_ids'],
        attention_mask = input_tensor['attention_mask'],
        max_length = 128
    )
output = output.cpu().numpy()
print(f"Input : {text}")
print(f"Translated : {tokenizer.decode(output,skip_special_tokens=True)}")

Input : Rohit sharma holds the record of highest runs by individual batsman that is 264 runs in a single inning
Translated : ['रोहिट sharma के पास व्यक्तिगत बल्लेबाजी के उच्चतम रनों की रिकार्ड है, जो एक पारी में 264 रन हैं।']
